<a href="https://colab.research.google.com/github/toddbalwinski/ds2002-fa26/blob/main/studios/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [4]:
# TODO
print(df.shape)

print(df.dtypes)

print(df.isna().sum())

print(df.duplicated().sum())

(8, 6)
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
1


**What is wrong with this data?** List at least five specific problems:

1. There is a row that is an exact duplicate
2. Order 3 is missing a quantity
3. Order 6 is missing a timestamp
4. Order 5 has a negative quantity
5. Order 7 is missing an item

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [8]:
removed = df.duplicated().sum()  # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

log('duplicates', 'dropped exact duplicate rows', removed)


[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [12]:
clean['price'] = (
    clean['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float))

assert clean['price'].dtype == float
log('price', 'stripped dollar signs and whitespace', len(clean))

[price] stripped dollar signs and whitespace (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [13]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities

# TODO: apply your decision, then log both separately

clean = clean.dropna(subset=['qty']).copy()
log('qty_missing', 'dropped rows with missing quantity', missing)
log('qty_negative', 'kept negative quantities', negative)

[qty_missing] dropped rows with missing quantity (1 row(s))
[qty_negative] kept negative quantities (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [18]:
print('before:', sorted(clean['category'].unique()))

before_count = clean['category'].nunique()

# TODO: lowercase, strip, remove punctuation
clean['category'] = (
    clean['category']
    .str.lower()
    .str.strip()
    .str.replace(r'[^\w\s]', '', regex=True)
)
# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {'raingear': 'rain gear'}
clean['category'] = clean['category'].replace(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))\

after_count = clean['category'].nunique()

log('categories', 'before vs after count', (before_count - after_count))

before: ['apparel', 'food', 'merch', 'rain gear']
after:  ['apparel', 'food', 'merch', 'rain gear']
[categories] before vs after count (0 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [19]:
# TODO
print('before:', sorted(clean['item'].dropna().unique()))

before_count = clean['item'].nunique(dropna=True)

clean['item'] = (
    clean['item']
    .str.lower()
    .str.strip())

ITEM_MAP = {'cheese burger': 'cheeseburger'}

clean['item'] = clean['item'].replace(ITEM_MAP)

missing_item = clean['item'].isna().sum()

clean = clean.dropna(subset=['item']).copy()

after_count = clean['item'].nunique(dropna=True)

print('after:', sorted(clean['item'].unique()))

log('items', 'before minus after counts', before_count - after_count)

log('item_missing', 'dropped rows with missing item name', missing_item)

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after: ['cheeseburger', 'rain poncho', 'uva t-shirt']
[items] before minus after counts (2 row(s))
[item_missing] dropped rows with missing item name (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [21]:
# TODO
clean['ts'] = pd.to_datetime(
    clean['ts'],
    errors='coerce',
    format='mixed')

failed = clean['ts'].isna().sum()

clean['hour'] = clean['ts'].dt.hour

log('timestamps','put timestamps to datetime and made failures to NaT', failed)

[timestamps] put timestamps to datetime and made failures to NaT (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [23]:
# TODO: assertions
assert clean.duplicated().sum() == 0
assert clean['price'].dtype == float
assert clean['qty'].notna().all()
assert clean['item'].notna().all()
assert clean['category'].notna().all()

clean['revenue'] = clean['qty'] * clean['price']

# TODO: print rows, units, revenue, distinct categories
print(len(clean))
print(clean['qty'].sum())
print(clean['revenue'].sum())
print(clean['category'].nunique())

5
6.0
76.5
3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [22]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,duplicates,dropped exact duplicate rows,1
2,price,stripped dollar signs and whitespace,7
3,qty_missing,dropped rows with missing quantity,1
4,qty_negative,kept negative quantities,1
5,categories,before vs after count,0
6,items,before minus after counts,2
7,item_missing,dropped rows with missing item name,1
8,timestamps,parsed timestamps to datetime and made failure...,1
9,timestamps,put timestamps to datetime and made failures t...,1


**The decision that mattered most:** was keeping the negative quantiy as a refund

**Revenue with it:** 76.50 **Revenue without it:** 94.5

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [24]:
# Checkpoint
rows_after = 5            # TODO
revenue_after = 76.5         # TODO
biggest_decision = 'Keeping negative revenue as a refund'    # TODO: which choice moved the number most
revenue_other_way = 94.5     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 76.5
decision that mattered: Keeping negative revenue as a refund
revenue the other way: 94.5
